# Entraînement du classifieur mammo bénin/malin — Colab GPU (variante **Google Drive**)

Identique au notebook HF, mais le dataset CBIS est lu depuis **Google Drive monté** (pas de
repo HF dataset requis). Pipeline : Drive → manifestes (split par patient stratifié) →
entraînement (`cropped` **et** `full`) → comparaison → publication de l'artefact versionné sur HF.

**Cibles** : **AUC ≥ 0,85**, **sensibilité ≥ 0,90**, spécificité ≥ 0,75.

**Pré-requis** : Runtime **GPU** ; ton Drive contient le dataset CBIS (images + CSV de description
de cas + éventuellement `dicom_info.csv`) ; un **token HF `write`** uniquement pour l'étape de
publication (l'entraînement n'en a pas besoin).

> ⚠️ Le modèle de production est le **cropped** (le `ml-service` classe des ROIs YOLO). Le **full**
> est entraîné pour comparaison. Ne publie/déploie rien tant que les cibles ne sont pas atteintes.

## 1. Paramètres (⚠️ adapte les chemins Drive à ton arborescence)

In [ ]:
# Dossier du dataset CBIS DANS ton Drive (une fois monté sous /content/drive/MyDrive/...)
DRIVE_DATA_DIR = '/content/drive/MyDrive/CBIS-DDSM'   # ← à adapter

# CSV de description de cas, RELATIFS à DRIVE_DATA_DIR :
CASE_CSVS = [
    'csv/mass_case_description_train_set.csv',
    'csv/calc_case_description_train_set.csv',
    'csv/mass_case_description_test_set.csv',
    'csv/calc_case_description_test_set.csv',
]
# 'csv/dicom_info.csv' si export Kaggle (mapping SeriesUID->jpeg) ; None si arbre DICOM->jpeg préservé.
DICOM_INFO = 'csv/dicom_info.csv'
# Racine des images (préfixe des chemins résolus). Souvent = DRIVE_DATA_DIR.
IMAGES_ROOT = DRIVE_DATA_DIR

HF_MODEL_REPO = 'Mailcoding/ifar-mammo-classifier'   # repo model HF privé (publication)
VERSION = 'v0.1.0'
USES = ['cropped', 'full']
VAL_FRAC, SEED = 0.2, 42

## 2. Installer le package `ifar-mlops`

In [ ]:
# Option A — dépôt dédié (une fois peuplé) :
!git clone https://github.com/mailcoding/ifar-mlops.git 2>/dev/null || true
# Option B (repli) — monorepo produit : clone puis %cd <repo>/mlops.
%cd /content/ifar-mlops
!pip install -e '.[train]' -q
import torch; print('CUDA dispo :', torch.cuda.is_available())

## 3. Monter Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_DIR = DRIVE_DATA_DIR
assert os.path.isdir(DATA_DIR), f'Introuvable : {DATA_DIR} — vérifie le chemin dans la cellule 1.'
print('Dataset Drive :', DATA_DIR)
!ls -la "{DATA_DIR}" | head

## 4. Générer les manifestes (split par patient stratifié) + class_weights

On utilise l'API `build()` pour récupérer les `class_weights` suggérés et vérifier l'absence de
fuite patient. `verify=True` contrôle que chaque fichier image existe bien sur le Drive.

In [ ]:
import os, csv
from mlops.datasets.build_cbis_manifest import build

def write_manifest(rows, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w', newline='') as f:
        w = csv.writer(f); w.writerow(['path', 'label'])
        for r in rows: w.writerow([r['path'], r['label']])

case_paths = [os.path.join(DATA_DIR, c) for c in CASE_CSVS]
dicom_info = os.path.join(DATA_DIR, DICOM_INFO) if DICOM_INFO else None
REPORTS = {}
for use in USES:
    rep = build(case_paths, use=use, dicom_info=dicom_info, images_root=IMAGES_ROOT,
                val_frac=VAL_FRAC, seed=SEED, verify=True)
    s = rep['stats']
    assert not s['patient_leak'], f'FUITE PATIENT ({use}) !'
    assert rep['train'] and rep['val'], (f"[{use}] train/val vide — vérifie DICOM_INFO/IMAGES_ROOT "
                                         f"(résolues {s['n_resolved']}, manquants {s['missing_files']}).")
    write_manifest(rep['train'], f'/content/manifests/{use}/train.csv')
    write_manifest(rep['val'],   f'/content/manifests/{use}/val.csv')
    REPORTS[use] = rep
    print(f"[{use}] résolues={s['n_resolved']} (manquants {s['missing_files']}) | "
          f"train {s['train']} / val {s['val']} | class_weights {s['suggested_class_weights']}")

## 5. Entraîner les deux variantes (`cropped` puis `full`)

In [ ]:
import yaml, copy, subprocess
base = yaml.safe_load(open('configs/mammo_classifier.yaml'))
for use in USES:
    cfg = copy.deepcopy(base)
    cfg['data']['train_manifest'] = f'/content/manifests/{use}/train.csv'
    cfg['data']['val_manifest']   = f'/content/manifests/{use}/val.csv'
    cfg['train']['class_weights'] = REPORTS[use]['stats']['suggested_class_weights']
    cfg['export']['out_dir'] = f'/content/artifacts/{use}'
    cfg['export']['version'] = VERSION
    p = f'/content/config_{use}.yaml'; yaml.safe_dump(cfg, open(p, 'w'))
    print(f'\n===== ENTRAÎNEMENT [{use}] (class_weights={cfg["train"]["class_weights"]}) =====')
    subprocess.run(['python', '-m', 'mlops.train.train_mammo_classifier', '--config', p], check=True)

## 6. Comparer les métriques

In [ ]:
import json
for use in USES:
    m = json.load(open(f'/content/artifacts/{use}/manifest.json'))['metrics']
    print(f"[{use:7}] AUC={m.get('auc')} sens={m.get('sensitivity')} spec={m.get('specificity')} "
          f"| @sens>={m.get('sensitivity_target')}: thr={m.get('operating_threshold_sens')} "
          f"spec={m.get('specificity_at_target_sens')}")
print('\nRappel cibles : AUC>=0.85, sensibilité>=0.90, spécificité>=0.75.')

## 7. Publier l'artefact de production (variante `cropped`)

Nécessite un **token HF `write`**. Ne publie que si les cibles sont atteintes.

In [ ]:
import os, json
from huggingface_hub import login
from mlops.registry import publish_artifact

os.environ['HF_TOKEN'] = 'hf_xxx'   # ← token HF write (publication uniquement)
login(os.environ['HF_TOKEN'])

PROD = 'cropped'
m = json.load(open(f'/content/artifacts/{PROD}/manifest.json'))['metrics']
ok = (m.get('auc') or 0) >= 0.85 and (m.get('sensitivity') or 0) >= 0.90
if not ok:
    print(f'⛔ Cibles non atteintes ({PROD}) — publication ANNULÉE. metrics={m}')
else:
    url = publish_artifact(f'/content/artifacts/{PROD}', HF_MODEL_REPO, VERSION)
    print('✅ Publié :', url, '(tag', VERSION, ')')

## 8. Suite (hors notebook)
- **Porte de validation clinique** (SaMD) avant `validated: true`.
- **Consommation produit** : le Space `ml-service` récupère la version (`pull_artifact` /
  `deploy-ml-space.yml`), nom de fichier `efficientnet_b0.pth`.
- **Suivi prod** : dashboard ops `/metrics`.
- ⚠️ Vider les sorties du notebook avant tout commit (aucune donnée patient dans git).